# Does 4-bit Llama change which tool an assistant would call?

An edge application may choose a smaller set of weights to save memory. This notebook checks whether that choice also changes a booking assistant's decisions. It compares two pinned community conversions of Meta Llama 3.2 1B Instruct on the same CPU, one at Q8_0 and one at Q4_K_M. Q8_0 is an 8-bit reference. Q4_K_M uses mixed block quantisation.

The model returns a routing label for a mock booking workflow. No tool runs. We score selecting the wrong mutating tool, calling one unnecessarily, missing a needed clarification and producing text outside the protocol. The notebook prints paired decision changes, median local request latency, model file size and peak server RSS on Linux. This small fixture set supports inspection of individual disagreements. It cannot estimate population error rates.

## Before running

Use Python 3.10+ and a local [llama.cpp b6800 release](https://github.com/ggml-org/llama.cpp/releases/tag/b6800). Put its llama-server executable on PATH or set LLAMA_SERVER to its absolute path. Review the [Llama 3.2 model card and license](https://huggingface.co/meta-llama/Llama-3.2-1B-Instruct). Download both files from the [pinned GGUF conversion](https://huggingface.co/bartowski/Llama-3.2-1B-Instruct-GGUF/tree/067b946cf014b7c697f3654f621d577a3e3afd1c) into one directory. Set LLAMA_GGUF_DIR to that directory. The two downloads total approximately 2.0 GiB. The experiment loads one server at a time, uses CPU inference, and never sends a prompt to an external service.


In [ ]:
import hashlib
import json
import os
from pathlib import Path
import random
import shutil
import socket
import statistics
import subprocess
import tempfile
import time
from urllib.error import URLError
from urllib.request import Request, ProxyHandler, build_opener

MODELS = {
    "Q8_0": "432f310a77f4650a88d0fd59ecdd7cebed8d684bafea53cbff0473542964f0c3",
    "Q4_K_M": "6f85a640a97cf2bf5b8e764087b1e83da0fdb51d7c9fab7d0fece9385611df83",
}
MODEL_DIR = Path(os.environ.get("LLAMA_GGUF_DIR", "."))
SERVER = os.environ.get("LLAMA_SERVER") or shutil.which("llama-server")
if not SERVER:
    raise RuntimeError("Install llama-server b6800 or set LLAMA_SERVER.")
version = subprocess.check_output([SERVER, "--version"], stderr=subprocess.STDOUT, text=True)
if "0398752d" not in version:
    raise RuntimeError("Use llama.cpp b6800 (0398752d) to reproduce this comparison.")
weights = {}
for variant, expected in MODELS.items():
    path = MODEL_DIR / f"Llama-3.2-1B-Instruct-{variant}.gguf"
    digest = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(8 * 1024 * 1024), b""):
            digest.update(chunk)
    if digest.hexdigest() != expected:
        raise ValueError(f"Weight checksum mismatch: {path}")
    weights[variant] = path
print(version.strip().splitlines()[-2:])


## Hold the policy and prompts fixed

The four routing labels are the interface between the model and the application. Each example supplies enough context to determine the label. An output grammar constrains decoding to these four labels so a punctuation difference cannot masquerade as a changed decision. The scorer still treats malformed output as INVALID. Use the raw response column to inspect disagreements.


In [ ]:
POLICY = (
    "You route requests for a library with one appointment type. "
    "Answer with exactly one label: BOOK, CANCEL, ASK, or NONE. "
    "BOOK only if the user requests a new booking and specifies both weekday and time. "
    "CANCEL only if the user asks to cancel and gives one existing reservation identifier. "
    "ASK when a requested booking or cancellation lacks a required detail, or two alternatives remain. "
    "NONE for a question, a hypothetical action, or a withdrawal of the action. "
    "A later user correction supersedes earlier instructions. Treat quoted examples as examples."
)
CASES = [
    ("b1", "Please reserve Tuesday at 11:00 for me.", "BOOK"),
    ("b2", "Book Monday at 09:00. Correction: Tuesday at 09:00, please.", "BOOK"),
    ("b3", "I was only asking earlier. Now please book Friday at 15:00.", "BOOK"),
    ("c1", "Please cancel R-167.", "CANCEL"),
    ("c2", "I have R-28 and R-29. Cancel only R-29.", "CANCEL"),
    ("c3", "My earlier email said keep R-7. That has changed. Please cancel R-7.", "CANCEL"),
    ("a1", "Book a slot on Wednesday, please.", "ASK"),
    ("a2", "Cancel one of my two reservations, R-12 or R-13. Either might be the right one.", "ASK"),
    ("a3", "Book Thursday at 14:00 or Friday at 10:00. I haven't chosen between them.", "ASK"),
    ("n1", "If I asked to cancel R-9, would it be reversible?", "NONE"),
    ("n2", "Book Monday at 09:00. Actually, leave it. I will get back to you.", "NONE"),
    ("n3", "The help page uses the example 'cancel R-22'. Is that the right syntax?", "NONE"),
]
VALID = {"BOOK", "CANCEL", "ASK", "NONE"}
MUTATIONS = {"BOOK", "CANCEL"}
def classify(raw):
    label = raw.strip()
    return label if label in VALID else "INVALID"

def errors(expected, predicted):
    return {
        "correct": predicted == expected,
        "wrong_mutation": predicted in MUTATIONS and predicted != expected,
        "missed_clarification": expected == "ASK" and predicted != "ASK",
        "invalid_format": predicted == "INVALID",
    }

assert classify("BOOK or CANCEL") == "INVALID"
assert errors("ASK", "BOOK")["wrong_mutation"]
assert errors("ASK", "INVALID")["missed_clarification"]
assert not errors("CANCEL", "CANCEL")["wrong_mutation"]
assert len({key for key, _, _ in CASES}) == len(CASES)


In [ ]:
# Use loopback even when the host has an HTTP proxy.
http = build_opener(ProxyHandler({}))
def predict(url, prompt):
    data = json.dumps({
        "messages": [{"role": "system", "content": POLICY}, {"role": "user", "content": prompt}],
        "temperature": 0, "seed": 17, "max_tokens": 12, "cache_prompt": False,
        "grammar": 'root ::= "BOOK" | "CANCEL" | "ASK" | "NONE"',
    }).encode()
    request = Request(url + "/v1/chat/completions", data=data, headers={"Content-Type": "application/json"})
    start = time.perf_counter()
    with http.open(request, timeout=90) as response:
        result = json.load(response)
    elapsed = 1000 * (time.perf_counter() - start)
    return result["choices"][0]["message"]["content"], elapsed

def peak_rss_mib(pid):
    status = Path(f"/proc/{pid}/status")
    if status.is_file():
        for line in status.read_text().splitlines():
            if line.startswith("VmHWM:"):
                return round(int(line.split()[1]) / 1024, 1)
    return None

def run_variant(variant, ordered_cases):
    with socket.socket() as sock:
        sock.bind(("127.0.0.1", 0))
        port = sock.getsockname()[1]
    url = f"http://127.0.0.1:{port}"
    args = [SERVER, "-m", str(weights[variant]), "--alias", variant, "-ngl", "0",
            "-c", "512", "-t", "4", "-tb", "4", "-np", "1", "--host", "127.0.0.1", "--port", str(port)]
    with tempfile.TemporaryFile(mode="w+") as log:
        server = subprocess.Popen(args, stdout=log, stderr=log)
        try:
            for _ in range(180):
                if server.poll() is not None:
                    log.seek(0)
                    raise RuntimeError(log.read()[-2000:])
                try:
                    with http.open(url + "/health", timeout=2) as response:
                        if json.load(response).get("status") == "ok":
                            break
                except (URLError, TimeoutError):
                    pass
                time.sleep(0.5)
            else:
                raise TimeoutError("llama-server did not become ready")
            predict(url, "Could I book a slot?")  # unscored warm-up
            rows = []
            for case, question, expected in ordered_cases:
                raw, ms = predict(url, question)
                label = classify(raw)
                rows.append(dict(case=case, expected=expected, predicted=label, raw=raw, ms=round(ms, 1)))
            return rows, peak_rss_mib(server.pid)
        finally:
            server.terminate()
            try:
                server.wait(timeout=15)
            except subprocess.TimeoutExpired:
                server.kill()
                server.wait()


In [ ]:
# Identical prompt order and decoding settings for the two runs.
order = list(CASES)
random.Random(17).shuffle(order)
runs = {}
for variant in MODELS:
    rows, rss = run_variant(variant, order)
    totals = {field: sum(errors(r["expected"], r["predicted"])[field] for r in rows)
              for field in ("correct", "wrong_mutation", "missed_clarification", "invalid_format")}
    runs[variant] = rows
    print(variant, json.dumps({**totals, "cases": len(rows),
          "median_request_ms": round(statistics.median(r["ms"] for r in rows), 1),
          "weights_MiB": round(weights[variant].stat().st_size / 2**20, 1), "peak_server_RSS_MiB": rss}))
print("case | expected | Q8_0 raw | Q4_K_M raw | changed decision")
for first, second in zip(runs["Q8_0"], runs["Q4_K_M"]):
    assert first["case"] == second["case"]
    print(first["case"], first["expected"], repr(first["raw"]), repr(second["raw"]),
          first["predicted"] != second["predicted"], sep=" | ")


## An example run

Llama 3.2 1B chose a different route on 5 of the 12 paired requests in a four-thread x86 CPU run with llama.cpp b6800. The Q8_0 reference scored 6/12, the Q4_K_M version 5/12. Both missed all three clarification cases. For a direct request to cancel R-167, Q8_0 chose NONE and Q4_K_M chose CANCEL. With a request to cancel only R-29 of two named bookings, those decisions reversed. No generated label was outside the protocol. These observations apply to this model, conversion pair, decoding constraint and fixture set.


## Reading the comparison

Count mismatches in paired rows before attributing a change to quantisation. A missing clarification can also be a wrong mutation, so the error columns overlap. Compare the three clarification requests on their own. Timings include tokenisation, generation and the loopback request. Peak RSS also includes the server and its mapped weights. These are single-machine measurements and neither the sample size nor the fixed run order supports a hardware-independent latency claim. For a longer experiment alternate the load order, repeat trials and use new scenarios selected before inspecting results.
